# Sun energy: combined historical data EDA

Explore **every file in `data/raw/sun` together**, following `1_co2_data_eda.ipynb`: schema, coverage, missing hours, duplicates, plausibility, distribution, and seasonality. Assess solar production as an input for the 168-hour CO₂ forecast in `docs/initial_assignment.md`.

Preserve raw values; flag unusual observations without deleting or imputing them. UTC defines the hourly grid; Europe/Amsterdam defines calendar profiles. Add solar-specific zero checks and a descriptive comparison with the historical CO₂ target. No model is fitted here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/initial_assignment.md").exists())
DATA_DIR = ROOT / "data/raw/sun"
TARGET = "volume (kWh)"

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.25})

## 1. Load every source file

The directory currently contains CSV exports. Fail visibly if an unexpected file format or schema appears, rather than silently omitting it. Keep source filenames to inspect boundaries and provenance. `validfrom` is the interval start and `validto` is its end; both headers explicitly specify UTC.

In [ ]:
files = sorted(p for p in DATA_DIR.iterdir() if p.is_file())
assert files, "No input files found"
assert all(p.suffix.lower() == ".csv" for p in files), "Review unsupported source files"

frames = [pd.read_csv(p) for p in files]
assert all(frame.columns.equals(frames[0].columns) for frame in frames), "Source schemas differ"

inventory = pd.DataFrame({"file": [p.name for p in files], "bytes": [p.stat().st_size for p in files], "rows": [len(f) for f in frames]})
df = pd.concat([f.assign(source_file=p.name) for p, f in zip(files, frames)], ignore_index=True)
raw_columns = list(frames[0].columns)

for col in ["validfrom (UTC)", "validto (UTC)"]:
    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
df = df.sort_values("validfrom (UTC)").reset_index(drop=True)

display(inventory)

In [ ]:
display(df.head())

In [ ]:
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "unique_values": df.nunique(dropna=False)}))

In [ ]:
print(f"Total: {len(df):,} rows, {len(raw_columns)} original columns")

`volume (kWh)` is energy generated during each interval and is the feature explored here. Charts convert it to GWh. `capacity (kW)` is a power-valued field: numerical agreement with one-hour energy does not establish installed capacity. The denominator of `percentage` is undocumented; do not treat it as solar share of the electricity mix.

The solar export's emissions and emission-factor fields are not the total-mix CO₂ target. Inspect metadata and constant fields; `Current` does not establish forecast issue times, publication latency, or revision history.

In [ ]:
metadata_columns = ["point", "type", "granularity", "timezone", "activity", "classification"]
display(pd.DataFrame({c: [df[c].unique().tolist()] for c in metadata_columns}).T.rename(columns={0: "observed_values"}))

In [ ]:
numeric_columns = df[raw_columns].select_dtypes(include="number").columns.tolist()
display(df[numeric_columns].describe(percentiles=[.01, .25, .5, .75, .99]).T)

In [ ]:
display(df.groupby("source_file").agg(rows=(TARGET, "size"), start_utc=("validfrom (UTC)", "min"), end_utc_exclusive=("validto (UTC)", "max"), solar_mean_kwh=(TARGET, "mean")))

## 2. Missing values, duplicate rows, and missing hours

A file can have no null cells while still omitting entire hours. Check both, including cross-file boundaries. Do not resolve duplicate timestamps silently. Coverage is measured only between the first and last observed timestamps; this cannot establish completeness outside that range.

In [ ]:
display(pd.DataFrame({"missing": df.isna().sum(), "missing_pct": df.isna().mean() * 100}))

In [ ]:
start = df["validfrom (UTC)"]
end = df["validto (UTC)"]
duplicate_times = start.duplicated(keep=False)
expected = pd.date_range(start.min(), start.max(), freq="h")
missing_hours = expected.difference(pd.DatetimeIndex(start.dropna()))
durations = (end - start).dt.total_seconds() / 3600
checks = pd.Series({
    "duplicate_raw_rows": df.duplicated(raw_columns).sum(),
    "rows_with_duplicate_start": duplicate_times.sum(),
    "invalid_start_or_end": (start.isna() | end.isna()).sum(),
    "non_hour_intervals": durations.ne(1).sum(),
    "off_hour_starts": start.ne(start.dt.floor("h")).sum(),
    "missing_hours_inside_coverage": len(missing_hours),
    "non_contiguous_adjacent_intervals": (start.iloc[1:].reset_index(drop=True) != end.iloc[:-1].reset_index(drop=True)).sum(),
})
display(checks.to_frame("count"))

In [ ]:
display(pd.DataFrame({"missing_hour_utc": missing_hours}).head(20))

In [ ]:
display(df.loc[duplicate_times].head(10))

In [ ]:
assert not duplicate_times.any() and start.notna().all(), "Resolve timestamp issues before time-series analysis"
s = df.set_index("validfrom (UTC)")[TARGET].reindex(expected)
s.index.name = "time_utc"
local = s.tz_convert("Europe/Amsterdam")
print(f"Local interval coverage: {start.min().tz_convert("Europe/Amsterdam")} to {end.max().tz_convert("Europe/Amsterdam")} (end exclusive)")

## 3. Plausibility and internal consistency

Check non-finite, negative, and zero values and compare energy with power × interval duration. Zero solar output can be expected at night; it is not a missing value. These exports alone cannot verify installed capacity or emissions accounting.

In [ ]:
display(pd.DataFrame({
    "non_finite": (~np.isfinite(df[numeric_columns])).sum(),
    "negative": df[numeric_columns].lt(0).sum(),
    "zero": df[numeric_columns].eq(0).sum(),
}))

In [ ]:
energy_error = df[TARGET] - df["capacity (kW)"] * durations
display(energy_error.describe().to_frame("volume_minus_capacity_times_hours_kwh"))

## 4. Distribution and possible outliers

Use 1.5 × IQR fences as descriptive flags, not physical thresholds. Solar is zero-inflated and seasonal, so large summer production can be valid. Flag absolute hourly changes above the 99th percentile; sunrise, sunset, and weather can cause large changes. Retain every observation.

In [ ]:
q1, q3 = df[numeric_columns].quantile(.25), df[numeric_columns].quantile(.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flags = df[numeric_columns].lt(lower) | df[numeric_columns].gt(upper)
display(pd.DataFrame({"lower_fence": lower, "upper_fence": upper, "flagged_rows": flags.sum(), "flagged_pct": flags.mean() * 100}))

In [ ]:
changes = s.diff().abs()
jump_threshold = changes.quantile(.99)
jump_flags = changes.gt(jump_threshold)
display(pd.DataFrame({"solar": s, "absolute_hourly_change": changes}).nlargest(10, "absolute_hourly_change"))

In [ ]:
fig, ax = plt.subplots()
ax.hist(s.dropna() / 1_000_000, bins=60)
ax.set(xlabel="Solar energy (GWh per hourly interval)", ylabel="Hours", title="Combined solar distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([g.dropna().to_numpy() / 1_000_000 for _, g in local.groupby(local.index.year)], tick_labels=[str(y) for y in sorted(local.index.year.unique())])
ax.set(xlabel="Local year (2026 partial)", ylabel="Solar energy (GWh per hourly interval)", title="Distribution by year")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(changes.dropna() / 1_000_000, bins=60)
ax.axvline(jump_threshold / 1_000_000, color="red", label="99th percentile")
ax.set(xlabel="Absolute hourly change (GWh)", ylabel="Hours", title="Changes between adjacent hours")
ax.legend()
plt.tight_layout()
plt.show()

### Zero output and long zero runs

Summarize zeros by local hour and inspect the longest contiguous UTC zero runs. A run spanning a whole day deserves review, but clock hours alone cannot identify daylight throughout the year. No astronomical daylight mask or automatic correction is applied.

In [ ]:
zero = s.eq(0)
zero_runs = zero.ne(zero.shift()).cumsum()
run_table = s.to_frame("solar_kwh").assign(run=zero_runs, zero=zero).groupby("run").agg(
    start_utc=("solar_kwh", lambda x: x.index.min()),
    end_utc_inclusive=("solar_kwh", lambda x: x.index.max()),
    hours=("solar_kwh", "size"), all_zero=("zero", "all"),
)
display(run_table.loc[run_table["all_zero"]].nlargest(10, "hours"))
zero_profile = local.to_frame("solar_kwh").groupby(local.index.hour)["solar_kwh"].agg(
    observations="count", zero_pct=lambda x: x.eq(0).mean() * 100,
)
display(zero_profile.rename_axis("local_hour"))

## 5. Time-series views

Plot hourly observations with daily and monthly means, the latest 28 days, and flagged jumps. Monthly means near dataset boundaries may cover partial months. Monthly coverage counts below make this visible.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index, s / 1_000_000, alpha=.25, linewidth=.4, label="Hourly")
ax.plot(s.resample("D").mean() / 1_000_000, linewidth=.8, label="Daily mean")
ax.plot(s.resample("MS").mean() / 1_000_000, linewidth=2, label="Monthly mean")
ax.scatter(s.index[jump_flags], s[jump_flags] / 1_000_000, s=8, color="red", label="Large hourly change")
ax.set(title="Combined history", ylabel="Solar energy (GWh per hourly interval)", xlabel="UTC")
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
recent = s.loc[s.index.max() - pd.Timedelta(days=28):]
ax.plot(recent.index, recent / 1_000_000)
ax.set(title="Latest 28 days", ylabel="Solar energy (GWh per hourly interval)", xlabel="UTC")
plt.tight_layout()
plt.show()

In [ ]:
display(local.resample("MS").agg(["count", "mean", "min", "max"]).rename_axis("local_month"))

## 6. Daily, weekly, and annual seasonality

Calendar profiles use Dutch local time, including daylight saving transitions. UTC stays regular; local spring days have 23 hours and autumn days 25. Compare years separately because 2026 is incomplete. Counts expose unequal coverage. Profiles show association, not a causal or stationary seasonal effect; fewer than three complete annual cycles limit annual conclusions.

In [ ]:
calendar = local.rename("solar").to_frame()
calendar["year"] = calendar.index.year
calendar["month"] = calendar.index.month
calendar["weekday"] = calendar.index.dayofweek
calendar["hour"] = calendar.index.hour
fig, ax = plt.subplots()

for year, group in calendar.groupby("year"):
    (group.groupby("hour")["solar"].mean() / 1_000_000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local hour", ylabel="Mean Solar energy (GWh per hourly interval)", title="Daily profile")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for year, group in calendar.groupby("year"):
    (group.groupby("weekday")["solar"].mean() / 1_000_000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local weekday", ylabel="Mean Solar energy (GWh per hourly interval)", title="Weekly profile (Monday = 0)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for year, group in calendar.groupby("year"):
    (group.groupby("month")["solar"].mean() / 1_000_000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local month", ylabel="Mean Solar energy (GWh per hourly interval)", title="Annual profile (2026 partial)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(calendar.groupby(["year", "month"])["solar"].agg(["count", "mean"]).unstack("year"))

In [ ]:
heatmap = calendar.pivot_table(index="weekday", columns="hour", values="solar", aggfunc="mean") / 1_000_000
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(heatmap, aspect="auto", origin="lower")
ax.set(xticks=range(24), yticks=range(7), yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], xlabel="Local hour", title="Mean CO₂ factor by weekday and hour")
fig.colorbar(im, ax=ax, label="GWh")
plt.tight_layout()
plt.show()

### Autocorrelation on a regular hourly grid

Compare raw autocorrelation with autocorrelation after subtracting a centered 30-day rolling mean to reduce slow variation. The centered smoother is for retrospective EDA only and would leak future information if used as a forecasting feature. Correlations use available aligned pairs without filling gaps. Peaks around 24 and 168 hours support daily and weekly recurrence but are not a formal seasonality test or evidence of forecast accuracy.

In [ ]:
detrended = s - s.rolling(24 * 30, center=True, min_periods=24 * 15).mean()
lags = np.arange(1, 24 * 14 + 1)
acf = pd.DataFrame({"raw": [s.autocorr(int(lag)) for lag in lags], "detrended": [detrended.autocorr(int(lag)) for lag in lags]}, index=lags)
ax = acf.plot(figsize=(12, 4))

for lag in [24, 168]:
    ax.axvline(lag, color="grey", linestyle="--", alpha=.7)
ax.set(xlabel="Lag (hours)", ylabel="Pearson autocorrelation", title="Hourly autocorrelation: daily and weekly recurrence")
plt.show()

In [ ]:
display(pd.DataFrame({"raw_correlation": [s.autocorr(lag) for lag in [1, 24, 168]], "detrended_correlation": [detrended.autocorr(lag) for lag in [1, 24, 168]], "available_pairs": [(s.notna() & s.shift(lag).notna()).sum() for lag in [1, 24, 168]]}, index=pd.Index([1, 24, 168], name="lag_hours")))

### Solar profile by month and hour

A month–hour heatmap exposes the changing width and height of the daily solar profile. Monthly energy totals are accompanied by observed and expected hours; partial months and partial years must not be compared as full periods.

In [ ]:
heatmap = calendar.pivot_table(index="month", columns="hour", values="solar", aggfunc="mean") / 1_000_000
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(heatmap, aspect="auto", origin="lower")
ax.set(xticks=range(24), yticks=range(len(heatmap)), yticklabels=heatmap.index, xlabel="Local hour", ylabel="Local month", title="Mean hourly solar energy")
fig.colorbar(im, ax=ax, label="GWh per hourly interval")
plt.tight_layout()
plt.show()
monthly = local.resample("MS").agg(observed_hours="count", energy_kwh=lambda x: x.sum(min_count=1))
monthly["expected_hours"] = [(t + pd.offsets.MonthBegin(1) - t).total_seconds() / 3600 for t in monthly.index]
monthly["coverage_pct"] = monthly["observed_hours"] / monthly["expected_hours"] * 100
monthly["energy_gwh"] = monthly.pop("energy_kwh") / 1_000_000
display(monthly)

## 7. Alignment with the historical CO₂ target

Join on UTC interval starts after checking unique one-hour intervals and matching interval ends. Report unmatched hours in both directions. Correlations are descriptive and confounded by season, demand, and other generation. Contemporaneous historical solar output is not known seven days in advance; deployment needs forecasts available at each forecast origin.

In [ ]:
co2_files = sorted((ROOT / "data/raw/co2").glob("*.csv"))
co2 = pd.concat([pd.read_csv(p) for p in co2_files], ignore_index=True)
for col in ["validfrom (UTC)", "validto (UTC)"]:
    co2[col] = pd.to_datetime(co2[col], utc=True, errors="raise")
assert co2["validfrom (UTC)"].is_unique, "Resolve duplicate CO2 hours before joining"
assert (co2["validto (UTC)"] - co2["validfrom (UTC)"]).eq(pd.Timedelta(hours=1)).all()
co2 = co2.set_index("validfrom (UTC)").sort_index()
solar_end = df.set_index("validfrom (UTC)")["validto (UTC)"]
common = s.index.intersection(co2.index)
assert solar_end.reindex(common).eq(co2.loc[common, "validto (UTC)"]).all()
paired = pd.concat([s.rename("solar_kwh"), co2["emissionfactor (kg CO2/kWh)"].rename("co2_factor")], axis=1).dropna()
display(pd.Series({"paired_hours": len(paired), "solar_hours_without_co2": len(s.dropna().index.difference(co2.index)), "co2_hours_without_solar": len(co2.index.difference(s.dropna().index))}).to_frame("count"))
correlations = pd.Series({"all_hours": paired["solar_kwh"].corr(paired["co2_factor"]), "positive_solar_hours": paired.loc[paired["solar_kwh"].gt(0), "solar_kwh"].corr(paired.loc[paired["solar_kwh"].gt(0), "co2_factor"])})
display(correlations.to_frame("Pearson_correlation"))
display(paired.groupby(paired.index.tz_convert("Europe/Amsterdam").year)[["solar_kwh", "co2_factor"]].corr().iloc[::2, 1].to_frame("yearly_correlation"))
fig, ax = plt.subplots()
im = ax.hexbin(paired["solar_kwh"] / 1_000_000, paired["co2_factor"] * 1000, gridsize=45, mincnt=1, bins="log")
ax.set(xlabel="Solar energy (GWh per hourly interval)", ylabel="CO₂ factor (g CO₂/kWh)", title="Historical solar production and total-mix carbon intensity")
fig.colorbar(im, ax=ax, label="log10(hour count)")
plt.tight_layout()
plt.show()

## 8. Findings and next steps

The summary below is computed from the input files. Consult the coverage tables and seasonal charts before interpreting changes between years.

In [ ]:
display(Markdown(f"""
- Loaded **{len(files)} files**, **{len(df):,} rows**; UTC interval coverage **{start.min()} – {end.max()}** (end exclusive).
- Missing raw cells: **{df[raw_columns].isna().sum().sum():,}**; missing interior hours: **{len(missing_hours):,}**; duplicate-start rows: **{duplicate_times.sum():,}**; non-hour intervals: **{durations.ne(1).sum():,}**.
- Hourly solar energy: **{s.min() / 1e6:.3f}–{s.max() / 1e6:.3f} GWh**, mean **{s.mean() / 1e6:.3f} GWh**; zero-output hours: **{zero.sum():,} ({zero.mean():.1%})**. Zeros are retained.
- Maximum absolute power–energy discrepancy: **{energy_error.abs().max():.3f} kWh**. Solar emission and factor columns each have **{df["emission (kg CO2)"].nunique()}** and **{df["emissionfactor (kg CO2/kWh)"].nunique()}** unique values; they do not replace the total-mix target.
- IQR flags: **{flags[TARGET].sum():,}**; large hourly changes: **{jump_flags.sum():,}** above **{jump_threshold / 1e6:.3f} GWh**. Flags are retained, not classified as errors.
- Autocorrelation at 24 / 168 hours: **{s.autocorr(24):.3f} / {s.autocorr(168):.3f}**. Daily recurrence is consistent with the solar cycle; weekly correlation can also reflect repeated daily and seasonal structure.
- Aligned solar–CO₂ Pearson correlation: **{correlations["all_hours"]:.3f}**, versus **{correlations["positive_solar_hours"]:.3f}** for positive solar output. This does not establish causality or forecast skill.
- **2026 is partial**; compare matching calendar periods and check monthly coverage before interpreting annual production differences.

Next: include solar interval energy in the core renewable feature set with onshore and offshore wind. Preserve nighttime zeros and UTC alignment; avoid redundant power and source-emission fields. Use actual generation as agreed: evaluation with future actuals assumes known production and excludes generation forecast error. Fit preprocessing on training data only and evaluate chronological 168-hour forecasts against persistence and repeated daily/weekly profiles. Archived forecasts are a future operational requirement rather than a prerequisite for this conditional experiment. See [the consolidated EDA report](../eda_report.md).
"""))